# 03 — RFM Analysis

## Objective
Engineer Recency, Frequency, and Monetary features from the cleaned 
transaction data, score each customer across all three dimensions, 
and assign meaningful segments to drive business decisions.

## Contents
1. Load cleaned data
2. Feature engineering — Recency, Frequency, Monetary
3. RFM scoring
4. Customer segmentation
5. Segment analysis

---

In [1]:
import datetime as dt
import pandas as pd

df = pd.read_csv('../data/processed/df_clean.csv')

In [2]:
print(df.shape)
df.head()

(392693, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Monetary Calculation

In [3]:
df['Total'] = df['Quantity'] * df['Price']
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
rfm = df[['Customer ID', 'Total']].groupby('Customer ID').sum()

In [5]:
rfm= rfm.reset_index()
rfm.columns = ['Customer ID', 'Monetary']

In [6]:
rfm.head()

,Customer ID,Monetary
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40


### Frequency Calculation

In [7]:
rfm = rfm.merge(df[['Customer ID','Invoice']].groupby('Customer ID').nunique().reset_index(), how = 'inner')

In [8]:
rfm = rfm.rename(columns={'Invoice':'Frequency'})
rfm.head()

,Customer ID,Monetary,Frequency
0,12346.0,77183.60,1
1,12347.0,4310.00,7
2,12348.0,1797.24,4
3,12349.0,1757.55,1
4,12350.0,334.40,1


### Recency Calculation

In [9]:
df['InvoiceDate'].dtype

<StringDtype(storage='python', na_value=nan)>

In [10]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [11]:
print(df['InvoiceDate'].max())
ref_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

2011-12-09 12:50:00


We can see that the last invoice date was on 09-12-2011. We will use the day after the last invoice date as the reference date to calculate the recency column.

In [12]:
recency = df[['Customer ID','InvoiceDate']].groupby('Customer ID').max().reset_index()
recency.head()

,Customer ID,InvoiceDate
0,12346.0,2011-01-18 10:01:00
1,12347.0,2011-12-07 15:52:00
2,12348.0,2011-09-25 13:13:00
3,12349.0,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00


In [13]:
recency['Recency'] = (ref_date - recency['InvoiceDate']).dt.days
recency.head()

,Customer ID,InvoiceDate,Recency
0,12346.0,2011-01-18 10:01:00,326
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,19
4,12350.0,2011-02-02 16:01:00,310


In [14]:
recency = recency.drop(['InvoiceDate'], axis = 1)
recency.head()

,Customer ID,Recency
0,12346.0,326
1,12347.0,2
2,12348.0,75
3,12349.0,19
4,12350.0,310


In [15]:
rfm = rfm.merge(recency, how='inner')

In [16]:
rfm.head()

,Customer ID,Monetary,Frequency,Recency
0,12346.0,77183.60,1,326
1,12347.0,4310.00,7,2
2,12348.0,1797.24,4,75
3,12349.0,1757.55,1,19
4,12350.0,334.40,1,310


Double Checking rfm table for valid values

In [17]:
df['Customer ID'].nunique()

4338

In [18]:
print(rfm.shape)
rfm.describe()

(4338, 4)


,Customer ID,Monetary,Frequency,Recency
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,2048.692230,4.272015,92.536422
std,1721.808492,8985.229676,7.697998,100.014169
min,12346.000000,3.750000,1.000000,1.000000
25%,13813.250000,306.482500,1.000000,18.000000
50%,15299.500000,668.570000,2.000000,51.000000
75%,16778.750000,1660.597500,5.000000,142.000000
max,18287.000000,280206.020000,209.000000,374.000000


In [32]:
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4])

In [36]:
rfm['F_Score'] = pd.qcut(rfm['Frequency'], q=4,labels= [1,2,3], duplicates='drop')

In [37]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels= [4,3,2,1])

In [38]:
rfm.head()

,Customer ID,Monetary,Frequency,Recency,M_Score,F_Score,R_Score
0,12346.0,77183.60,1,326,4,1,1
1,12347.0,4310.00,7,2,4,3,4
2,12348.0,1797.24,4,75,4,2,2
3,12349.0,1757.55,1,19,4,1,3
4,12350.0,334.40,1,310,2,1,1


In [40]:
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm.head()

,Customer ID,Monetary,Frequency,Recency,M_Score,F_Score,R_Score,RFM_Score
0,12346.0,77183.60,1,326,4,1,1,114
1,12347.0,4310.00,7,2,4,3,4,434
2,12348.0,1797.24,4,75,4,2,2,224
3,12349.0,1757.55,1,19,4,1,3,314
4,12350.0,334.40,1,310,2,1,1,112


### Assigning Segments to Customers

In [51]:
# Custom Function to assign categories based on RFM Score:
def assign_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    
    # Conditions for each category:
    if r==4 and f==3 and m>=3: # Best values is every category
        return 'Champion'
    if (r==3 or r==4) and (f==2 or f==3): # Spends frequently and recently
        return 'Loyal'
    if r==4 and f==1 and m<=2: #Bought less but very recent, new customer assumed
        return 'New Customer'
    if (r==3 or r==4): # Bought recently but low on how often they buy (f==1)
        return 'Potential Loyalist'
    if r==2 and f<=3: # Low Recency
        return 'At Risk'
    if r==1: # Worst Recency score
        return 'Lost'
    
    return 'Other' # If no condition matches.
    

In [45]:
rfm['Segment'] = rfm.apply(assign_segment,axis=1)
rfm.head()

,Customer ID,Monetary,Frequency,Recency,M_Score,F_Score,R_Score,RFM_Score,Segment
0,12346.0,77183.60,1,326,4,1,1,114,Lost
1,12347.0,4310.00,7,2,4,3,4,434,Champion
2,12348.0,1797.24,4,75,4,2,2,224,At Risk
3,12349.0,1757.55,1,19,4,1,3,314,Potential Loyalist
4,12350.0,334.40,1,310,2,1,1,112,Lost


In [47]:
rfm['Segment'].value_counts()

Segment
Lost                  1084
Loyal                  988
At Risk                972
Potential Loyalist     532
Champion               461
New Customer           207
Other                   94
Name: count, dtype: int64

There are still customers showing up as other, will fix that first.

In [49]:
rfm[rfm['Segment'] == 'Other'][['R_Score','F_Score','M_Score']].value_counts()

R_Score  F_Score  M_Score
2        3        4          69
                  3          24
                  2           1
                  1           0
         1        4           0
                  3           0
                  2           0
                  1           0
         2        4           0
                  3           0
                  2           0
                  1           0
4        3        4           0
                  3           0
                  2           0
                  1           0
         1        4           0
                  3           0
                  2           0
                  1           0
         2        4           0
                  3           0
                  2           0
                  1           0
3        3        4           0
                  3           0
                  2           0
                  1           0
         1        4           0
                  3           0
              

Reapplying changed function

In [52]:
rfm['Segment'] = rfm.apply(assign_segment,axis=1)
rfm.head()

,Customer ID,Monetary,Frequency,Recency,M_Score,F_Score,R_Score,RFM_Score,Segment
0,12346.0,77183.60,1,326,4,1,1,114,Lost
1,12347.0,4310.00,7,2,4,3,4,434,Champion
2,12348.0,1797.24,4,75,4,2,2,224,At Risk
3,12349.0,1757.55,1,19,4,1,3,314,Potential Loyalist
4,12350.0,334.40,1,310,2,1,1,112,Lost


In [53]:
rfm['Segment'].value_counts()

Segment
Lost                  1084
At Risk               1066
Loyal                  924
Potential Loyalist     532
Champion               525
New Customer           207
Name: count, dtype: int64

No Other count anymore, means segmenting function accounts for all cases

Segment Analysis

In [54]:
rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2)

,Recency,Frequency,Monetary
Segment,,,
At Risk,84.31,2.64,1030.20
Champion,7.48,15.10,8459.53
Lost,246.98,1.63,650.63
Loyal,23.30,5.32,2278.01
New Customer,10.09,1.53,318.55
Potential Loyalist,30.58,1.49,886.61


In [56]:
rfm[['Recency','Frequency','Monetary']].mean()

Recency        92.536422
Frequency       4.272015
Monetary     2048.692230
dtype: float64

Looking at the mean values for each segment we can infer the following:

**At Risk** (n = 1066 Customers)
- Data: Last purchased an average of 84 days ago, with an average of 2.64 orders and $1030 total spent.
- Interpretation: These customers showed interest initially but interest is fading. Low frequency and declining recency confirms customers have not returned for purchases. At risk of becoming lost if not re-engaged soon.
- Recommendation: Launch a win-back campaign with a time-limited discount or personalized product recommendation based on past purchases. This will re-engage the customers. Priority target before they cross into lost territory.

**Champion** (n = 525 Customers)
- Data: Last purchased an average of 7 days ago, with an average of 15 orders and total spend of $8459.
- Interpretation: These are returning, high spending customers who are completely engaged in our products. 
- Recommendation: Launch a survey to find out what products these customers are interested in. Priority to refine these products to continue keeping these customers engaged. In addition, launch a loyalty program that awards returning customers with points or discounts as appropriate.

**Lost** (n = 1084 Customers)
- Data: Customers with a last purchased average of 246 days, an order average of 1.63 and $650 total spend.
- Interpretation: These customers have showed low frequency and engagement from the start, and never returned.
- Recommendation: Research products these customers bought and improve them. These customers are very hard to re-engage and might not be worth catching their attention. Research what made them leave and fix the issues to avoid losing any new customers.

**Loyal** (n = 924 Customers)
- Data: Last purchased an average of 23 days ago, with an average of 5 orders and $2275 total spent.
- Interpretation: Customers who frequently re-engage with the products, however they still have a significant gap in frequency to the 'Champion' Customers.
- Recommendation: Loyal customers will come back as long as the product they want will remain available. However, an investigation should be run to check what type of products they buy and launch an advertisement campaign on similar products to help increase their engagement. This will help increase their purchases which can help these customers move up to the 'Champion' Category. Priority as these are the majority of the customers where company revenue comes from.

**Potential Loyalist** (n = 532 Customers)
- Data: Last Purchased an average of 30 days ago, with an average of 1.5 orders and $886 spent.
- Interpretation: These customers are showing signs of becoming 'Loyal' Customers, and are slightly older new customers who have minimal new engagement. Chances of falling into the 'At Risk' category if not re-engaged. 
- Recommendation: Launch advertisement campaigns periodically for the most bought products by customers in this category. This will remind them of our products and push them to return. In addition, targetted second-purchase incentives like a discount on their next order will help retain them better. Priority to increase customer base.

**New Customer** (n = 207 Customers)
- Data: Last purchased an average of 10 days ago, with an average of 1.53 orders and a total spent of $318.
- Interpretation: These are customers who have only ordered once or twice very recently. Likely customers who have just bought the product and are yet to have a reason to return.
- Recommendation: Monitor the products bought by these new customers as they are most likely the face of the company, as their experience impacts the reputation of the company. Having better first impressions will encorage new customers to return and spread the company through word of mouth, leading to free marketing. Keep track of any customers who might not re-engage and launch advertisement campaigns to re-engage them as needed. Track any reviews or returns made more recently and address those issues in order to stop engagement from fading.